In [1]:
# Project Variables

lakehouse_name = "lh_dev_ecommerce"
env_name = "dev"
job_id = 104

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 3, Finished, Available, Finished, False)

In [2]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_silver_db
""")

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
customers_df = spark.table(
    f"{lakehouse_name}.{env_name}_stage_db.customers"
)

# customers_df.show(5)

# customers_df.printSchema()

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import to_date

customers_df = customers_df.withColumn(
    "SignupDate",
    to_date("SignupDate", "yyyy-MM-dd")
)

# customers_df.printSchema()

# customers_df.show(5)

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 6, Finished, Available, Finished, False)

In [5]:
from pyspark.sql.functions import upper, trim

customers_df = customers_df.withColumn(
    "IsPremiumCustomer",
    upper(trim("IsPremiumCustomer"))
)

customers_df.select("IsPremiumCustomer").distinct()

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 7, Finished, Available, Finished, False)

DataFrame[IsPremiumCustomer: string]

In [6]:
before_count = customers_df.count()

customers_df = customers_df.dropDuplicates(["CustomerID"])

after_count = customers_df.count()

print("Before:", before_count)
print("After :", after_count)
print("Duplicates Removed:", before_count - after_count)

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 8, Finished, Available, Finished, False)

Before: 28
After : 28
Duplicates Removed: 0


In [7]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {lakehouse_name}.{env_name}_silver_db.customers
(
    CustomerID STRING,
    CustomerName STRING,
    IsPremiumCustomer STRING,
    SignupDate DATE,
    InsertTimestamp TIMESTAMP
)
USING DELTA
""")

print("Silver customers table created successfully.")

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 9, Finished, Available, Finished, False)

Silver customers table created successfully.


In [8]:
customers_df.createOrReplaceTempView("customers_stage")

print("Temporary view created.")

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 10, Finished, Available, Finished, False)

Temporary view created.


In [9]:
# spark.sql(f"""
# SELECT OrderID, OrderDate
# FROM {lakehouse_name}.{env_name}_stage_db.orders
# LIMIT 10
# """).show(truncate=False)

StatementMeta(, 77ed3715-aef7-496c-87d7-22768f9ac4bd, 11, Finished, Available, Finished, False)